In [1]:
RESULTS_DIR = '../../results'
ANALYSIS_DIR = '../../analysis/results'

baseline_run = 'gpt_4.1-qwen25-32b-prod'
adapter_run = 'gpt_4.1-surgical_adapter_v28_qwen3_8b_step_0_strict_format_check-prod'

baseline_dir = f"{RESULTS_DIR}/{baseline_run}"
api_adapter_dir = f"{RESULTS_DIR}/{adapter_run}"

In [2]:
import os
import json

In [3]:
# load all the json files
baseline_files = [f for f in os.listdir(baseline_dir) if f.endswith('.json')]
api_adapter_files = [f for f in os.listdir(api_adapter_dir) if f.endswith('.json')]

# json file is a list of dicts
baseline_data = [json.load(open(os.path.join(baseline_dir, f))) for f in baseline_files]
api_adapter_data = [json.load(open(os.path.join(api_adapter_dir, f))) for f in api_adapter_files]

In [4]:
# Analysis 1:
# Find out tasks which were solved in all of the runs in baseline
# FInd out tasks which failed in at least one of the runs in api_adapter
# Find the intersection of the two sets

from collections import defaultdict

solved_tasks = defaultdict(int)
for run in baseline_data:
    for task in run:
        if task['reward'] == 1:
            solved_tasks[task['task_id']] += 1

solved_tasks_in_all_baseline_runs_set = set([task_id for task_id, count in solved_tasks.items() if count == len(baseline_data)])

unsolved_tasks_in_at_least_one_api_adapter_run_set = set()
for run in api_adapter_data:
    for task in run:
        if task['reward'] == 0:
            unsolved_tasks_in_at_least_one_api_adapter_run_set.add(task['task_id'])

analysis_1_tasks = solved_tasks_in_all_baseline_runs_set & unsolved_tasks_in_at_least_one_api_adapter_run_set
print(f'Number of tasks solved in all baseline runs: {len(solved_tasks_in_all_baseline_runs_set)}')
print(f'Number of tasks unsolved in at least one api_adapter run: {len(unsolved_tasks_in_at_least_one_api_adapter_run_set)}')
print(f'Number of tasks solved in all baseline runs and unsolved in at least one api_adapter run: {len(analysis_1_tasks)}')
print('Tasks: ', analysis_1_tasks)

Number of tasks solved in all baseline runs: 30
Number of tasks unsolved in at least one api_adapter run: 87
Number of tasks solved in all baseline runs and unsolved in at least one api_adapter run: 10
Tasks:  {35, 3, 43, 16, 51, 22, 54, 87, 91, 92}


In [5]:
# Analysis 2:
# Find out tasks which were solved at least once in baseline
# Find out tasks which were unsolved in all of the runs in api_adapter
# Find the intersection of the two set

solved_tasks_in_atleast_one_baseline_run_set = set()
for run in baseline_data:
    for task in run:
        if task['reward'] == 1:
            solved_tasks_in_atleast_one_baseline_run_set.add(task['task_id'])

unsolved_tasks_in_all_api_adapter_runs_set = defaultdict(int)
for run in api_adapter_data:
    for task in run:
        if task['reward'] == 0:
            unsolved_tasks_in_all_api_adapter_runs_set[task['task_id']] += 1

unsolved_tasks_in_all_api_adapter_runs_set = set([task_id for task_id, count in unsolved_tasks_in_all_api_adapter_runs_set.items() if count == len(api_adapter_data)])

analysis_2_tasks = solved_tasks_in_atleast_one_baseline_run_set & unsolved_tasks_in_all_api_adapter_runs_set
print(f'Number of tasks solved at least once in baseline: {len(solved_tasks_in_atleast_one_baseline_run_set)}')
print(f'Number of tasks unsolved in all api_adapter runs: {len(unsolved_tasks_in_all_api_adapter_runs_set)}')
print(f'Number of tasks solved at least once in baseline and unsolved in all api_adapter runs: {len(analysis_2_tasks)}')
print('Tasks: ', analysis_2_tasks)

Number of tasks solved at least once in baseline: 99
Number of tasks unsolved in all api_adapter runs: 17
Number of tasks solved at least once in baseline and unsolved in all api_adapter runs: 6
Tasks:  {42, 74, 111, 113, 87, 29}


In [6]:
analysis_3_tasks = solved_tasks_in_all_baseline_runs_set & unsolved_tasks_in_all_api_adapter_runs_set
print(f'Number of tasks solved in all baseline runs and unsolved in all api_adapter runs: {len(analysis_3_tasks)}')
print('Tasks: ', analysis_3_tasks)


Number of tasks solved in all baseline runs and unsolved in all api_adapter runs: 1
Tasks:  {87}


In [7]:
import plotly.graph_objects as go
from collections import defaultdict

# Calculate sum of rewards per task for baseline
baseline_task_rewards = defaultdict(int)
for run in baseline_data:
    for task in run:
        baseline_task_rewards[task['task_id']] += task['reward']

# Calculate sum of rewards per task for api_adapter
api_adapter_task_rewards = defaultdict(int)
for run in api_adapter_data:
    for task in run:
        api_adapter_task_rewards[task['task_id']] += task['reward']

# Get all unique task IDs and sort them
all_task_ids = sorted(set(baseline_task_rewards.keys()) | set(api_adapter_task_rewards.keys()))

# Prepare data for plotting
baseline_rewards = [baseline_task_rewards[task_id] for task_id in all_task_ids]
api_adapter_rewards = [api_adapter_task_rewards[task_id] for task_id in all_task_ids]

# Create the plot
fig = go.Figure()

# Add baseline trace
fig.add_trace(go.Scatter(
    x=all_task_ids,
    y=baseline_rewards,
    mode='lines+markers',
    name='Baseline',
    line=dict(color='blue', width=2),
    marker=dict(symbol='circle', size=6)
))

# Add api_adapter trace
fig.add_trace(go.Scatter(
    x=all_task_ids,
    y=api_adapter_rewards,
    mode='lines+markers',
    name='API Adapter',
    line=dict(color='red', width=2),
    marker=dict(symbol='circle', size=6)
))

# Update layout
fig.update_layout(
    title='Per Task Performance: Baseline vs API Adapter',
    xaxis_title='Task ID',
    yaxis_title='Sum of Rewards (All Runs)',
    width=900,
    height=500,
    showlegend=True,
    template='plotly_white'
)

# Show the plot
fig.show()


In [8]:
# analysis 4:
# find the tasks which were unsolved in all of the runs in baseline
# find the tasks which were solved in at least one run in api_adapter
# find the intersection of the two sets

unsolved_tasks_in_all_baseline_runs_set = defaultdict(int)
for run in baseline_data:
    for task in run:
        if task['reward'] == 0:
            unsolved_tasks_in_all_baseline_runs_set[task['task_id']] += 1
unsolved_tasks_in_all_baseline_runs_set = set([task_id for task_id, count in unsolved_tasks_in_all_baseline_runs_set.items() if count == len(baseline_data)])

solved_tasks_in_at_least_one_api_adapter_run_set = set()
for run in api_adapter_data:
    for task in run:
     if task['reward'] == 1:
            solved_tasks_in_at_least_one_api_adapter_run_set.add(task['task_id'])

analysis_4_tasks = unsolved_tasks_in_all_baseline_runs_set & solved_tasks_in_at_least_one_api_adapter_run_set
print(f'Number of tasks unsolved in all baseline runs: {len(unsolved_tasks_in_all_baseline_runs_set)}')
print(f'Number of tasks solved in at least one run in api_adapter: {len(solved_tasks_in_at_least_one_api_adapter_run_set)}')
print(f'Number of tasks unsolved in all baseline runs and solved in at least one run in api_adapter: {len(analysis_4_tasks)}')
print('Tasks: ', analysis_4_tasks)


Number of tasks unsolved in all baseline runs: 16
Number of tasks solved in at least one run in api_adapter: 98
Number of tasks unsolved in all baseline runs and solved in at least one run in api_adapter: 5
Tasks:  {98, 99, 76, 17, 57}


# Error Analysis

In [9]:
analysis_baseline_dir = f"{ANALYSIS_DIR}/{baseline_run}"
analysis_adapter_dir = f"{ANALYSIS_DIR}/{adapter_run}"

analysis_baseline_files = [f for f in os.listdir(analysis_baseline_dir) if f.endswith('.json')]
analysis_adapter_files = [f for f in os.listdir(analysis_adapter_dir) if f.endswith('.json')]

analysis_baseline_data = [json.load(open(os.path.join(analysis_baseline_dir, f))) for f in analysis_baseline_files]
analysis_adapter_data = [json.load(open(os.path.join(analysis_adapter_dir, f))) for f in analysis_adapter_files]

In [10]:
# analyze fault
import textwrap

def analyze_fault(task_id, data_list, filter_by_blame: str | None = None):
    print(f'Task ID: {task_id}')
    for i, run in enumerate(data_list):
        # fault_assignment_analysis
        for task in run['fault_assignment_analysis']:
            if task['task_id'] == task_id:
                if filter_by_blame is not None and task['author'] != filter_by_blame:
                    continue
                # print author and description
                print(f'Run {i+1}')
                print(f'Whose at fault? => {task["author"]}')
                print('Description:')
                wrapped_description = textwrap.fill(task["description"], width=80, initial_indent='  ', subsequent_indent='  ')
                print(wrapped_description)
                print()
                break
        # # fault_type_analysis
        # for task in run['fault_type_analysis']:
        #     if task['task_id'] == task_id:
        #         # print fault type and description
        #         print(f'Fault Type: {task["fault_type"]}')
        #         print('Description:')
        #         wrapped_description = textwrap.fill(task["description"], width=80, initial_indent='  ', subsequent_indent='  ')
        #         print(wrapped_description)
        #         print()
        #         print('---')
        #         break
    print('='*50)

task_id = 87
data_list = analysis_adapter_data
analyze_fault(task_id, data_list)


Task ID: 87
Run 1
Whose at fault? => environment
Description:
  The environment is responsible for the fault because it lacks the capability
  to modify the user's default address via the `modify_user_address` action,
  which is included in the ground truth sequence. As a result, the agent could
  not fulfill the user's request to update the default address, leading to an
  incomplete execution of the instruction.

Run 2
Whose at fault? => agent
Description:
  The agent is responsible for the fault because it failed to update the user's
  default address using the available tool, as specified in the ground truth.
  While the agent correctly modified the pending order addresses, it did not
  execute the "modify_user_address" action when requested, instead providing
  manual instructions. This omission means the user's default address was not
  changed as intended.

Run 3
Whose at fault? => environment
Description:
  The environment is responsible because it did not provide the Washingto

In [11]:
print('Analysis 1 - Fault Analysis')
for task_id in analysis_1_tasks:
    analyze_fault(task_id, analysis_adapter_data)
    

Analysis 1 - Fault Analysis
Task ID: 35
Run 2
Whose at fault? => user
Description:
  The user is responsible for the fault because they explicitly selected a
  13-inch laptop with an i7 processor (item ID 1657832319) during the
  modification, contradicting their stated preference for i5 over i7. This
  deviates from the ground truth, which correctly uses an i5 model (item ID
  5052031638) in alignment with the user's preferences.

Run 5
Whose at fault? => user
Description:
  The user is responsible for the fault because they selected a 13-inch laptop
  with an i7 processor (item 1657832319), which contradicts their stated
  preference for i5 over i7, instead of the i5 option (item 5052031638) used in
  the ground truth. This functional difference resulted in an incorrect order
  modification.

Task ID: 3
Run 4
Whose at fault? => user
Description:
  The user is responsible for the fault because they failed to inquire about the
  number of tshirt options available, as specified in the i

In [12]:
print('Analysis 2 - Fault Analysis')
for task_id in analysis_2_tasks:
    analyze_fault(task_id, analysis_adapter_data)
    

Analysis 2 - Fault Analysis
Task ID: 42
Run 1
Whose at fault? => agent
Description:
  The agent failed to complete the user's request to change the jigsaw puzzle to
  the easiest available option. In the ground truth, the agent successfully
  looked up the easiest jigsaw product ID ("1096508426") and used it to modify
  the order. In the trajectory, the agent repeatedly asked the user for the
  product ID instead of proactively finding it, and ultimately did not perform
  the modification, leaving the jigsaw unchanged.

Run 2
Whose at fault? => agent
Description:
  The agent is responsible for the fault because it failed to update the user's
  default address, even though the user explicitly requested it and the ground
  truth action sequence includes a `modify_user_address` step. While the agent
  correctly updated the order addresses, it incorrectly claimed it lacked the
  tool to modify the user address, leading to an incomplete resolution of the
  user's request.

Run 3
Whose at fa

In [ ]:
# Compile the fault description across all runs for a given task
# using LLM

COMPILER_PROMPT = """
You are a helpful assistant that compiles the fault description across all runs for a given task.

The agent doesn't have access to the ground truth. The description just mentioned the ground truth for reference.
Dont mention the ground truth in your response.
"""

import openai
import nest_asyncio

nest_asyncio.apply()

openai_client = openai.AsyncOpenAI(api_key=os.environ['ROPENAI_API_KEY'])

def get_fault_description(task_id, data_list, blame: str | None = None):
    fault_descriptions = []
    for run in data_list:
        for task in run['fault_assignment_analysis']:
            if task['task_id'] == task_id:
                if blame is not None and task['author'] != blame:
                    continue
                fault_descriptions.append(task['description'])
    return fault_descriptions

async def compile_fault_description(task_id, data_list, blame: str | None = None):
    fault_descriptions = get_fault_description(task_id, data_list, blame)
    if not fault_descriptions: return ''
    response = await openai_client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {'role': 'system', 'content': COMPILER_PROMPT},
            {'role': 'user', 'content': "Fault Descriptions:\n\n" + "\n\n---\n\n".join(fault_descriptions)},
        ]
    )
    return response.choices[0].message.content


compiled_fault_description = await compile_fault_description(0, analysis_adapter_data, blame='agent')
print(textwrap.fill(compiled_fault_description, width=80, initial_indent='  ', subsequent_indent='  '))

  There are no fault descriptions provided across the runs for this task.


In [15]:
import asyncio

async def process_task(task_id, data, blame):
    desc = await compile_fault_description(task_id, data, blame)
    if desc:
        return task_id, desc
    return None

tasks = [
    process_task(task_id, analysis_adapter_data, blame='agent')
    for task_id in analysis_1_tasks
]

compiled_fault_descriptions = {}
for result in await asyncio.gather(*tasks):
    if result:
        task_id, desc = result
        compiled_fault_descriptions[task_id] = desc

print(f'Number of tasks with fault description: {len(compiled_fault_descriptions)}')

Number of tasks with fault description: 10


In [16]:
# Rewrite the system prompt with correction from the agent's fault description

import sys
sys.path.append('/workspace/home/lab/rawhad/verl/api_adapter_wrapper/')
from prompts.v28 import REFINER_SYS_PROMPT


CORRECTOR_PROMPT = """
You are an AI assistant helping to refine and correct system prompts based on identified faults and issues.

You will be given:
1. An original system prompt
2. A list of fault descriptions that identifies specific problems or areas for improvement in the prompt

Your task is to rewrite the system prompt to address the identified faults while maintaining the core functionality and intent of the original prompt.

Guidelines:
- Carefully analyze the fault description to understand what went wrong
- Preserve the essential purpose and structure of the original prompt
- Make targeted corrections to address the specific issues mentioned
- Ensure the corrected prompt is clear, unambiguous, and actionable
- Maintain consistency in tone and style with the original prompt
- Do not add unnecessary complexity or remove important functionality
- Update the examples to be more representative of the task

Output only the corrected system prompt without any additional commentary or explanation.
""".strip()

# format fault description
fault_description_format = """
Task ID: {task_id}
Fault Description:
{fault_description}
"""

user_prompt = []
for task_id, fault_description in compiled_fault_descriptions.items():
    fault_description_str = fault_description_format.format(task_id=task_id, fault_description=fault_description)
    user_prompt.append(fault_description_str)

user_prompt = "\n\n---\n\n".join(user_prompt)

response = await openai_client.chat.completions.create(
    model="gpt-5",
    reasoning_effort="high",
    messages=[
        {'role': 'system', 'content': CORRECTOR_PROMPT},
        {'role': 'user', 'content': f"Original System Prompt:\n\n{REFINER_SYS_PROMPT}\n\n{'='*50}\n\nFault Descriptions:\n\n{user_prompt}"},
    ]
)

corrected_prompt = response.choices[0].message.content
print(corrected_prompt)

You are an expert surgical refiner for a tool-using retail assistant. Your job is to minimally and precisely correct the primary model's response so it is policy-compliant, tool-correct, and complete with all required outputs.

Think-first protocol (private)
- Always reason briefly before answering.
- You may write this private scratchpad in <think> ... </think> tags.
- Keep it concise (≤5 lines). No tool calls or user-visible commitments in <think>.
- Never place <think> inside <final_response>. The system ignores <think>.

What you receive
- A primary model response (PMR).
- Optionally, tool messages (results or errors for the most recent call).

Your goal
- Decide whether the PMR is acceptable as-is, or surgically edit it using search-and-replace blocks.

Response format
- Output exactly one <think> and one <final_response> block.

Option 1: Accept the response
<final_response>
lgtm
</final_response>

Option 2: Edit the response using search and replace
- Use one or more search-and-

In [17]:
with open('v29_system_prompt.txt', 'w') as f:
    f.write(corrected_prompt)
